In [0]:
from pyspark.sql.functions import floor, col, current_date, datediff, avg

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)

## Q1


For this question, I worked directly with the pit_stops dataset since it already records pit stop times in milliseconds.

First, I needed to make sure the milliseconds column could be used for calculations, so I converted it from string to integer.

Then I grouped the data by raceId and driverId so that each row represents one driver in one race.

After grouping, I calculated three values for each group: the average pit stop time, the slowest (maximum) time, and the fastest (minimum) time.

This gives a clear summary of pit stop performance for each driver in each race.


In [0]:
from pyspark.sql.functions import col, avg, max, min

df_pit_stops = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/pit_stops.csv",
    header=True
)

df_pit_stops_clean = df_pit_stops.withColumn(
    "milliseconds",
    col("milliseconds").cast("int")
)

df_q1 = df_pit_stops_clean.groupBy("raceId", "driverId").agg(
    avg("milliseconds").alias("avg_pit_stop_time"),
    max("milliseconds").alias("slowest_pit_stop_time"),
    min("milliseconds").alias("fastest_pit_stop_time")
)

display(df_q1.orderBy("raceId", "driverId"))


### Code explanation

I first loaded the `pit_stops` dataset.

Then I converted the `milliseconds` column to integer so it can be used in calculations.

After that, I grouped the data by `raceId` and `driverId`, so each group represents one driver in one race.

Within each group, I used `avg()` to get the average pit stop time, `max()` for the slowest stop, and `min()` for the fastest one.

I renamed the columns using `alias()` so the result is easier to read, and then displayed it.


## Extra credit:
One alternative approach would be to use Spark SQL instead of the DataFrame API. After creating a temporary view from the pit_stops dataset, I could write a SQL query with GROUP BY raceId and driverId and use AVG, MAX, and MIN directly.

Another option would be to join this result with the drivers dataset to include driver names instead of only driverId, which would make the output easier to interpret.

## Q2


For this question, I needed two pieces of information: pit stop time and finishing position.

I used the `pit_stops` dataset to calculate the average pit stop time for each driver in each race. Then I used the `results` dataset to get each driver’s finishing position in that same race.

After that, I joined the two results together by `raceId` and `driverId`. Once both values were in the same table, I ranked drivers within each race based on finishing position.

This gives a table where drivers are ordered by where they finished, along with their average pit stop time in that race.


In [0]:
from pyspark.sql.functions import col, avg
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)


df_pit_avg = df_pit_stops.withColumn(
    "milliseconds",
    col("milliseconds").cast("int")
).groupBy(
    "raceId", "driverId"
).agg(
    avg("milliseconds").alias("avg_pit_stop_time")
)


df_finish = df_results.select(
    "raceId",
    "driverId",
    "positionOrder"
).withColumn(
    "positionOrder",
    col("positionOrder").cast("int")
)

df_q2_base = df_pit_avg.join(
    df_finish,
    on=["raceId", "driverId"],
    how="inner"
)


race_window = Window.partitionBy("raceId").orderBy(col("positionOrder").asc())

df_q2 = df_q2_base.withColumn(
    "finish_rank",
    row_number().over(race_window)
).orderBy("raceId", "finish_rank")

display(df_q2)


### Code explanation

This part uses both the `pit_stops` and `results` datasets.

From `pit_stops`, I converted `milliseconds` to integer and grouped by `raceId` and `driverId` to get the average pit stop time.

From `results`, I kept `raceId`, `driverId`, and `positionOrder`, and converted the position to integer.

Then I joined the two tables so each row has both the average pit stop time and the finishing position.

To order drivers within each race, I used a window with `partitionBy("raceId")` and sorted by `positionOrder`. Then `row_number()` assigns the ranking.

The final result shows drivers in finishing order together with their average pit stop time.



### Extra credit

Another way to rank drivers would be to use `rank()` or `dense_rank()` instead of `row_number()` in case there are ties.

It would also be helpful to join the drivers dataset so the result shows names instead of only `driverId`.

Another extension would be to check whether faster pit stop times are associated with better finishing positions.


## Q3

In the drivers dataset, I noticed that some values in the code column are missing and shown as \N.

To fill these in, I used the driver’s surname. Most driver codes are just the first three letters of the last name (for example, Alonso → ALO, Hamilton → HAM), so I followed the same idea here.

For rows where the code is missing, I took the first three letters of the surname and converted them to uppercase. If a driver already had a code, I kept it as it is.

I saved the result in a new column instead of replacing the original one, so it’s easier to compare and check the results.

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
from pyspark.sql.functions import col, when, upper, substring, regexp_replace


df_drivers_code = df_drivers.withColumn(
    "code_filled",
    when(
        col("code") == "\\N",
        upper(
            substring(
                regexp_replace(col("surname"), "[^A-Za-z]", ""),
                1,
                3
            )
        )
    ).otherwise(col("code"))
)

display(
    df_drivers_code.select(
        "driverId", "forename", "surname", "code", "code_filled"
    )
)


### Code explanation

I created a new column called `code_filled` using `withColumn()` so the original column is still kept.

Then I used `when()` to find rows where the code is missing (`\N`).

For those rows, I cleaned the surname using `regexp_replace()`, took the first three letters with `substring()`, and converted them to uppercase.

If the code already exists, `otherwise()` keeps it unchanged.

At the end, I selected a few columns and displayed them to check the result.



### Extra credit

Another way to fill the missing codes would be to use `driverRef` instead of the surname, since it is usually cleaner.

Also, this method could create duplicate codes for drivers with similar last names, so one more step could be added to check for duplicates.

It would also be possible to compare the generated codes with an official driver code list if one is available.


## Q4

For this question, I wanted to find the youngest and oldest driver in each race.

The first thing I had to decide was how to define age. Instead of using today’s date, I used the race date, since it makes more sense to measure a driver’s age at the time of the race.

To do this, I joined the drivers dataset with the races dataset, so I could have both the date of birth and the race date in the same table. Then I calculated age by taking the difference between those two dates and converting it into years.

After that, I made sure each driver only appears once per race by using distinct, since lap times have multiple rows per driver.

Finally, I used window functions to rank drivers by age within each race. One ranking gives me the youngest driver, and the other gives me the oldest.

In [0]:
from pyspark.sql.functions import col, floor, datediff, row_number
from pyspark.sql.window import Window


df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)


df_race_drivers = df_laptimes.select("raceId", "driverId").distinct()


df_age_base = df_race_drivers.join(
    df_drivers.select("driverId", "forename", "surname", "dob", "nationality"),
    on="driverId",
    how="inner"
).join(
    df_races.select("raceId", "name", "date"),
    on="raceId",
    how="inner"
).withColumn(
    "Age",
    floor(datediff(col("date"), col("dob")) / 365.25)
)


youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())

df_youngest = df_age_base.withColumn(
    "youngest_rank",
    row_number().over(youngest_window)
).filter(col("youngest_rank") == 1).select(
    "raceId", "name", "date", "driverId", "forename", "surname", "nationality", "Age"
).withColumnRenamed("driverId", "youngest_driverId") \
 .withColumnRenamed("forename", "youngest_forename") \
 .withColumnRenamed("surname", "youngest_surname") \
 .withColumnRenamed("nationality", "youngest_nationality") \
 .withColumnRenamed("Age", "youngest_age")


oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

df_oldest = df_age_base.withColumn(
    "oldest_rank",
    row_number().over(oldest_window)
).filter(col("oldest_rank") == 1).select(
    "raceId", "driverId", "forename", "surname", "nationality", "Age"
).withColumnRenamed("driverId", "oldest_driverId") \
 .withColumnRenamed("forename", "oldest_forename") \
 .withColumnRenamed("surname", "oldest_surname") \
 .withColumnRenamed("nationality", "oldest_nationality") \
 .withColumnRenamed("Age", "oldest_age")

df_q4 = df_youngest.join(df_oldest, on="raceId", how="inner").orderBy("raceId")

display(df_q4)


### Code explanation

I first loaded the `races` dataset to get the race date.

Then I created a table of drivers in each race using `lap_times` and removed duplicates with `distinct()`.

After joining with the `drivers` and `races` datasets, I calculated age using `datediff()` and divided by 365.25, then applied `floor()`.

To find the youngest and oldest drivers, I used window functions. One sorts age ascending to get the youngest, and the other sorts descending to get the oldest.

Finally, I joined the two results so each race shows both values.



### Extra credit

Another option would be to use the `results` dataset instead of `lap_times` to get the list of drivers in each race.

Also, the age calculation here is an approximation. A more exact version could compare the full birth date with the race date.

It could also be interesting to keep all drivers and mark who was the youngest and who was the oldest in each race.


## Q5


For this question, I used the results dataset since podium finishes are based on final race results rather than lap-by-lap data.

I treated a podium as finishing 1st, 2nd, or 3rd. Then I created three separate columns to track these outcomes: wins, second places, and third places.

To track this over time, I ordered races by date for each driver and used cumulative sums. This way, for any given race, I can see how many podium finishes that driver had up to that point.

Finally, I combined these into a total podium count.

So the result shows, for each driver at each race, how many podiums they had accumulated at that stage.


In [0]:
from pyspark.sql.functions import col, when, sum
from pyspark.sql.window import Window


df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

df_results_base = df_results.select(
    "raceId", "driverId", "positionOrder"
)

df_race_info = df_races.select(
    "raceId", "date", "name"
)

df_q5_base = df_results_base.join(
    df_race_info,
    on="raceId",
    how="inner"
).join(
    df_drivers.select("driverId", "forename", "surname"),
    on="driverId",
    how="left"
)

df_q5_flags = df_q5_base.withColumn(
    "win_flag",
    when(col("positionOrder").cast("int") == 1, 1).otherwise(0)
).withColumn(
    "second_flag",
    when(col("positionOrder").cast("int") == 2, 1).otherwise(0)
).withColumn(
    "third_flag",
    when(col("positionOrder").cast("int") == 3, 1).otherwise(0)
)

driver_window = Window.partitionBy("driverId").orderBy(
    col("date").asc(), col("raceId").cast("int").asc()
).rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_q5 = df_q5_flags.withColumn(
    "wins",
    sum("win_flag").over(driver_window)
).withColumn(
    "second_places",
    sum("second_flag").over(driver_window)
).withColumn(
    "third_places",
    sum("third_flag").over(driver_window)
).withColumn(
    "podiums",
    col("wins") + col("second_places") + col("third_places")
).select(
    "raceId",
    "date",
    "name",
    "driverId",
    "forename",
    "surname",
    "wins",
    "second_places",
    "third_places",
    "podiums"
).orderBy("raceId", "driverId")

display(df_q5)


### Code explanation

I used the `results` dataset since it already contains finishing positions.

Then I joined it with the `races` dataset to bring in the race date, and with `drivers` to show names.

I created three columns using `when()` to mark whether a driver finished 1st, 2nd, or 3rd.

After that, I used a window partitioned by `driverId` and ordered by race date. This lets me calculate running totals using `sum().over()`.

These running totals give the number of wins, second places, and third places at each race.

Finally, I added them together to get total podiums.



### Extra credit

One variation would be to count podiums only before the current race, instead of including the current one.

It could also be useful to add more race information, like the year or circuit, to make the result easier to read.

Another idea would be to compare drivers within the same race based on how many podiums they had at that point.


## Q6

I wanted to look at how many drivers are in each race.

This is a simple question, but it helps understand the dataset a bit better. Some races might have more drivers than others, and this could affect things like competition level.

To answer this, I used the results dataset and counted how many unique drivers appear in each race.

In [0]:
from pyspark.sql.functions import countDistinct

df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

df_q6 = df_results.groupBy("raceId").agg(
    countDistinct("driverId").alias("num_drivers")
).join(
    df_races.select("raceId", "name", "date"),
    on="raceId",
    how="left"
).orderBy("raceId")

display(df_q6)


### Code explanation

I used the `results` dataset since it lists all drivers in each race.

Then I grouped by `raceId` and used `countDistinct()` to count how many drivers were in each race.

After that, I joined with the `races` dataset to add the race name and date.

Finally, I sorted the result and displayed it.



### Extra credit

A simple extension would be to calculate the average number of drivers per race.

It could also be interesting to check whether the number of drivers changed over time.

Another option would be to look for races with especially high or low driver counts.
